In [0]:
from pyspark.sql.functions import col, sum, count, rank, max as spark_max, current_date, expr
from pyspark.sql.window import Window

# 1. Apontar para o catálogo correto antes de ler as tabelas!
spark.sql("USE CATALOG cinedata_analytics")

# Carregar tabelas da camada Gold
df_fato = spark.read.table("gold.fact_movies_performance")
df_movies = spark.read.table("gold.dim_movies")
df_genres = spark.read.table("gold.dim_genres")
df_bridge_genre = spark.read.table("gold.bridge_movie_genre")
df_people = spark.read.table("gold.dim_people")
df_bridge_person = spark.read.table("gold.bridge_movie_person")
df_companies = spark.read.table("gold.dim_companies")
df_bridge_company = spark.read.table("gold.bridge_movie_company")

print("--- 1. Qual é a receita total (em R$) somada de todos os filmes da base? ---")
df_q1 = df_fato.select(sum("receita_brl").alias("receita_total_brl"))
display(df_q1)

print("--- 2. Quais são os 5 filmes com maior popularidade? ---")
df_q2 = df_movies.join(df_fato, "sk_movie_id", "inner") \
                 .select("titulo", "popularidade") \
                 .orderBy(col("popularidade").desc()) \
                 .limit(5)
display(df_q2)

print("--- 3. Quantos filmes cada gênero possui? (Do maior para o menor) ---")
df_q3 = df_bridge_genre.join(df_genres, "sk_genre_id", "inner") \
                       .groupBy("nome_genero") \
                       .agg(count("sk_movie_id").alias("qtd_filmes")) \
                       .orderBy(col("qtd_filmes").desc())
display(df_q3)

print("--- 4. Top 10 filmes de maior receita com a posição no ranking (RANK) ---")
window_rank = Window.orderBy(col("receita_usd").desc())
df_q4 = df_movies.join(df_fato, "sk_movie_id", "inner") \
                 .filter(col("receita_usd").isNotNull()) \
                 .withColumn("ranking", rank().over(window_rank)) \
                 .select("titulo", "receita_usd", "receita_brl", "ranking") \
                 .orderBy("ranking") \
                 .limit(10)
display(df_q4)


# --- Preparação para Perguntas 5 e 6 ---
# Identificar a data limite superior (data de lançamento mais recente ignorando o futuro)
data_limite_row = df_movies.filter(col("data_lancamento") <= current_date()) \
                           .select(spark_max("data_lancamento")).collect()[0][0]

print("--- 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos? ---")
# Filtro: data_lancamento >= (data_limite - 2 anos)
df_q5 = df_movies.filter(col("data_lancamento") >= expr(f"DATE '{data_limite_row}' - INTERVAL 2 YEARS")) \
                 .join(df_bridge_person, "sk_movie_id", "inner") \
                 .join(df_people, "sk_person_id", "inner") \
                 .filter(col("tipo_pessoa") == "Ator") \
                 .groupBy("nome_pessoa") \
                 .agg(count("sk_movie_id").alias("qtd_participacoes")) \
                 .orderBy(col("qtd_participacoes").desc()) \
                 .limit(1)
display(df_q5)

print("--- 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos? ---")
# Filtro: data_lancamento >= (data_limite - 5 anos)
df_q6 = df_movies.filter(col("data_lancamento") >= expr(f"DATE '{data_limite_row}' - INTERVAL 5 YEARS")) \
                 .join(df_fato, "sk_movie_id", "inner") \
                 .join(df_bridge_company, "sk_movie_id", "inner") \
                 .join(df_companies, "sk_company_id", "inner") \
                 .groupBy("nome_produtora") \
                 .agg(sum("lucro_usd").alias("lucro_total_usd")) \
                 .orderBy(col("lucro_total_usd").desc()) \
                 .limit(1)
display(df_q6)

--- 1. Qual é a receita total (em R$) somada de todos os filmes da base? ---


receita_total_brl
857060188984.47


--- 2. Quais são os 5 filmes com maior popularidade? ---


titulo,popularidade
Asylum of the Devil,2.0093201020121E12
A Postcard from Pyongyang,2.0132017E7
"The psychopath, chronicle of an unsolved case",1.9851995E7
Russian Hackers: The Beginning,902010.0
Mississippi Madam: The Life of Nellie Jackson,601990.0


--- 3. Quantos filmes cada gênero possui? (Do maior para o menor) ---


...

titulo,receita_usd,receita_brl,ranking
Avengers: Endgame,2800000000.00,14311080000.00,1
Avatar: The Way of Water,2320250281.00,11859031211.22,2
AVENGERS: INFINITY WAR,2052415039.00,10490098505.83,3
spider-man: no way home,1921847111.00,9822752769.03,4
The Lion King,1663075401.00,8500144682.05,5
Top Gun: Maverick,1488732821.00,7609062321.41,6
Barbie,1428545028.00,7301436492.61,7
The Super Mario Bros. Movie,1355725263.00,6929247391.72,8
Black Panther,1349926083.00,6899607202.82,9
Star Wars: The Last Jedi,1332698830.00,6811556990.01,10


--- 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos? ---


nome_pessoa,qtd_participacoes
Kevin Hart,62


--- 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos? ---


nome_produtora,lucro_total_usd
Universal Pictures,5790316738.00
